In [15]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
VALID_GENRES    = ["Action", "Animation", "Comedy", "Drama",
                   "Sci-Fi", "Thriller"]
VALID_MOODS     = ["Heartwarming", "Inspiring", "Lighthearted",
                   "Thought-provoking", "Thrilling"]
VALID_LANGUAGES = ["English", "Hindi", "Japanese",
                   "Korean", "Spanish", "Telugu"]

In [16]:
def load_data(path="movies_dataset.csv"):
    try:
        df = pd.read_csv(path)
        return df
    except FileNotFoundError:
        print(f"\n❌  Error: '{path}' not found.")
        print("    Make sure the CSV file is in the same folder as this script.")
        raise SystemExit(1)

In [17]:
def build_feature_soup(df):
    df = df.copy()
    df["features"] = (
        (df["genre"]    + " ") * 3
        + (df["mood"]   + " ") * 2
        + df["language"] + " "
        + df["description"]
    )
    return df

In [18]:
def vectorize_features(df):
    tfidf = TfidfVectorizer(stop_words="english")
    feature_matrix = tfidf.fit_transform(df["features"])
    return tfidf, feature_matrix

In [19]:
def _prompt(field_name, valid_options):
    hint = " / ".join(valid_options)
    while True:
        raw   = input(f"Preferred {field_name} ({hint}): ").strip()
        value = raw.title()
        if value in valid_options:
            return value
        print(f"  ⚠  '{raw}' is not recognised. Please choose from the list above.\n")
def get_user_preferences():
    print("\n--- Tell us what you like! ---")
    genre    = _prompt("genre",    VALID_GENRES)
    mood     = _prompt("mood",     VALID_MOODS)
    language = _prompt("language", VALID_LANGUAGES)
    return genre, mood, language

In [20]:
def recommend(genre, mood, language, df, tfidf, feature_matrix, top_n=5):
    user_text   = f"{genre} {genre} {genre} {mood} {mood} {language}"
    user_vector = tfidf.transform([user_text])

    scores = cosine_similarity(user_vector, feature_matrix).flatten()

    df = df.copy()
    df["match_score"] = (scores * 100).round(2)

    results = (
        df.sort_values(by=["match_score", "rating"], ascending=[False, False])
          .head(top_n)
    )
    return results[["title", "genre", "mood", "language",
                    "year", "rating", "match_score", "description"]]

In [21]:
def print_recommendations(results):
    print("\n" + "=" * 57)
    print("            🎬  RECOMMENDED FOR YOU  🎬")
    print("=" * 57)

    no_match = results.empty or results["match_score"].max() == 0
    if no_match:
        print("  No close matches found. Showing top-rated movies instead.\n")

    for rank, (_, row) in enumerate(results.iterrows(), start=1):
        print(f"\n  #{rank}  {row['title']}  ({row['year']})")
        print(f"       {row['genre']} · {row['mood']} · {row['language']}")
        print(f"       ⭐ IMDb: {row['rating']}  |  🔍 Match: {row['match_score']}%")
        print(f"       📝 {row['description']}")

In [22]:
def main():
    print("   🎥  AI MOVIE RECOMMENDATION SYSTEM  🎥")
    print("   Content-Based Filtering · Cosine Similarity")
    df                   = load_data("movies_dataset.csv")
    df                   = build_feature_soup(df)
    tfidf, feat_matrix   = vectorize_features(df)
    while True:
        genre, mood, language = get_user_preferences()
        results = recommend(genre, mood, language,
                            df, tfidf, feat_matrix, top_n=5)
        print_recommendations(results)
        again = input("\nWould you like another recommendation? (yes / no): ").strip().lower()
        if again not in ("yes", "y"):
            print("\nThanks for using the AI Recommendation System. Goodbye! 👋\n")
            break
if __name__ == "__main__":
    main()

   🎥  AI MOVIE RECOMMENDATION SYSTEM  🎥
   Content-Based Filtering · Cosine Similarity

--- Tell us what you like! ---
Preferred genre (Action / Animation / Comedy / Drama / Sci-Fi / Thriller): Action
Preferred mood (Heartwarming / Inspiring / Lighthearted / Thought-provoking / Thrilling): Thrilling
Preferred language (English / Hindi / Japanese / Korean / Spanish / Telugu): Telugu

            🎬  RECOMMENDED FOR YOU  🎬

  #1  Avengers: Endgame  (2019)
       Action · Thrilling · English
       ⭐ IMDb: 8.4  |  🔍 Match: 71.09%
       📝 The Avengers assemble for a final stand against Thanos.

  #2  Baahubali: The Beginning  (2015)
       Action · Thrilling · Telugu
       ⭐ IMDb: 8.0  |  🔍 Match: 68.61%
       📝 A young man uncovers his royal destiny and rises to reclaim his kingdom from a tyrant.

  #3  Baahubali 2: The Conclusion  (2017)
       Action · Thrilling · Telugu
       ⭐ IMDb: 9.7  |  🔍 Match: 68.33%
       📝 The truth behind a betrayal is revealed as a son rises to reclaim h